
# 🎬 Netflix Data Engineering Architecture

## 📦 Storage Configuration

| Component | Value |
|------------|---------|
| **Storage Account** | `vjweustg01` |
| **Landing Container** | `vj-landing` |
| **Landing Path** | `vj-landing/landing/netflix/` |

---

## 🏗️ Unity Catalog Structure

| Layer | Catalog / Schema |
|---------|----------------|
| 🥉 Bronze | `bronze` |
| 🥈 Silver | `silver` |
| 🥇 Gold | `gold` |

---

## 🎯 Target Schemas

| Purpose | Target |
|----------|---------|
| **Bronze Target** | `bronze.netflix` |
| **Metadata Target** | `bronze.metadata` |

---

### 📊 Data Flow

```text
Storage Account (vjweustg01)
        │
        ▼
Landing Container (vj-landing)
        │
        ▼
landing/netflix/
        │
        ▼
Bronze (bronze.netflix)
        │
        ▼
Silver
        │
        ▼
Gold
```

Problem are we solving? 

Netflix send us data every day. 

Day1 : 
    netflix_movie1.csv
    netflix_movie2.csv
    netflix_movie3.csv
    <br/>
Day2: 
    netflix_movie1.csv
    netflix_movie2.csv
    ntflix_movie3.csv
### How do we automatically load only new files into Databricks Bronze layer tables without realoading old files? 

### spark.read.csv();

Day1 =100 

Day 2 = Reads Day1 + Day2 <br>

Day3 = Reads Day1 + Day2 + Day3 

**X more compute Cost <br>
X slower processing <br>
X Duplicate records <br>
X Difficult auditing** 

# AUTO LOADER ?

# Delta Tables ?

ACID

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS bronze.netflix;
CREATE SCHEMA IF NOT EXISTS bronze.metadata;

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS ext_vj_landing_netflix
URL 'abfss://vj-landing@vjweustg01.dfs.core.windows.net/vj-landing/landing/netflix/'
WITH (STORAGE CREDENTIAL `uc-metastore-databricks_azuremanagedidentity_1779945538378`
);

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS ext_vj_bronze_metadata
URL 'abfss://vj-landing@vjweustg01.dfs.core.windows.net/vj-lakehouse/metadata/bronze/'
WITH (STORAGE CREDENTIAL `uc-metastore-databricks_azuremanagedidentity_1779945538378`);

In [0]:
%sql
CREATE EXTERNAL VOLUME IF NOT EXISTS bronze.metadata.landing_files
LOCATION 'abfss://vj-landing@vjweustg01.dfs.core.windows.net/vj-landing/landing/netflix/';

In [0]:
%sql
CREATE EXTERNAL VOLUME IF NOT EXISTS bronze.metadata.bronze_schemas
LOCATION 'abfss://vj-landing@vjweustg01.dfs.core.windows.net/vj-lakehouse/metadata/bronze/schemas/';

In [0]:
%sql
CREATE EXTERNAL VOLUME IF NOT EXISTS bronze.metadata.bronze_checkpoints
LOCATION 'abfss://vj-landing@vjweustg01.dfs.core.windows.net/vj-lakehouse/metadata/bronze/checkpoints/';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS bronze.metadata.bronze_config
(
  source_name STRING,
  dataset_name STRING,
  source_path STRING,
  file_pattern STRING,
  target_table STRING,
  delimiter STRING,
  header_flag STRING,
  is_active BOOLEAN,
  created_timestamp TIMESTAMP
);

In [0]:
%sql
INSERT INTO bronze.metadata.bronze_config VALUES
('netflix','netflix_tv_shows_movies',
'/Volumes/bronze/metadata/landing_files/',
'*/Netflix TV Shows and Movies.csv',
'bronze.netflix.netflix_tv_shows_movies',
',','true',true,current_timestamp()),

('netflix','netflix_stock_history',
'/Volumes/bronze/metadata/landing_files/',
'*/Netflix_stock_history.csv',
'bronze.netflix.netflix_stock_history',
',','true',true,current_timestamp()),

('netflix','netflix_movies',
'/Volumes/bronze/metadata/landing_files/',
'*/netflix_movies.csv',
'bronze.netflix.netflix_movies',
',','true',true,current_timestamp()),

('netflix','netflix_reviews',
'/Volumes/bronze/metadata/landing_files/',
'*/netflix_reviews.csv',
'bronze.netflix.netflix_reviews',
',','true',true,current_timestamp()),

('netflix','netflix_titles',
'/Volumes/bronze/metadata/landing_files/',
'*/netflix_titles.csv',
'bronze.netflix.netflix_titles',
',','true',true,current_timestamp()),

('netflix','netflix_tv_shows_detailed',
'/Volumes/bronze/metadata/landing_files/',
'*/netflix_tv_shows_detailed_up_to_2025 .csv',
'bronze.netflix.netflix_tv_shows_detailed',
',','true',true,current_timestamp());

In [0]:
%sql
CREATE TABLE IF NOT EXISTS bronze.metadata.bronze_audit_log
(
  run_id STRING,
  source_name STRING,
  dataset_name STRING,
  target_table STRING,
  status STRING,
  start_time TIMESTAMP,
  end_time TIMESTAMP,
  error_message STRING
);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS bronze.metadata.bronze_file_discovery
(
  source_name STRING,
  discovered_file_path STRING,
  discovered_file_name STRING,
  ingest_dt STRING,
  discovery_status STRING,
  discovered_timestamp TIMESTAMP,
  comments STRING
);